In [1]:
from dlfs.layers import EmbeddingLayer, PositionalEncoding, DenseLayer, LayerNorm
from dlfs.modules import TransformerDecoder

from dlfs.activation import Softmax, ReLU, GELU, Sigmoid
from dlfs.loss import CCE_Loss, BCE_Loss, MSE_Loss
from dlfs.optimizers import Optimizer_Adam

from dlfs.base import Module

import numpy as np

In [2]:
def get_random_batch(X, y, batch_size):
    idx = np.random.randint(0, len(X), size=(batch_size, ))
    return X[idx], y[idx]


In [3]:
class TransformerDecoderModel(Module):

    def __init__(self, vocab_size=12, block_size=5, 
                       n_embed=512, n_head = 4, n_dec_layers=2, dim_ff=2048,
                       dropout=0.1, activation=ReLU(), layer_norm_eps=1e-5, loss_function=None, optimizer=None):
        
        self.loss_function = loss_function
        self.optimizer = optimizer

        self.embed = EmbeddingLayer(vocab_size, n_embed)
        self.pos_enc = PositionalEncoding(block_size, n_embed)

        self.decoder = TransformerDecoder(d_model=n_embed, n_head=n_head, n_dec_layers=n_dec_layers,
                                          dim_ff=dim_ff, dropout=dropout, activation=activation, layer_norm_eps=layer_norm_eps)
        
        self.ln = LayerNorm(n_embed, epsilon=layer_norm_eps)

        self.linear = DenseLayer(n_embed, vocab_size)
        self.softm = Softmax()

    def forward(self, inputs, training):

        self.embed.forward(inputs, training)

        #sprint(f'embed out: {self.embed.output.shape}')

        self.pos_enc.forward(self.embed.output, training)

        self.decoder.forward(self.pos_enc.output, training)

        self.ln.forward(self.decoder.output, training)

        self.linear.forward(self.ln.output, training)

        self.softm.forward(self.linear.output, training)

        self.output = self.softm.output

    def backward(self, output, y):

        B, T, C = output.shape
        output = output.reshape(B*T, C)
        y = y.reshape(B*T, )

        #print(f'output: {output.shape}, y: {y.shape}')

        # Compute loss gradient
        self.loss_function.backward(output, y)
        
        # Backprop through softmax + linear
        self.softm.backward(self.loss_function.dinputs)
        self.linear.backward(self.softm.dinputs)

        self.ln.backward(self.linear.dinputs)

        # Backprop through decoder, positional encoding, and embedding
        self.decoder.backward(self.ln.dinputs)
        self.pos_enc.backward(self.decoder.dinputs)
        self.embed.backward(self.pos_enc.dinputs)

    def train(self, X, y, epochs = 1000, batch_size: int = None, print_every: int = None):

        self.loss_vals = []

        for i in range(epochs + 1):

            if batch_size is not None:

                batch_X, batch_y = get_random_batch(X, y, batch_size=batch_size)

                self.forward(batch_X, training=True)
                self.backward(self.output, batch_y)

                self.optimizer.pre_update_parameters()
                self.optimizer.update_parameters(self)
                self.optimizer.post_update_parameters()

                B, T, C = self.output.shape
                output = self.output.reshape(B*T, C)
                y_reshape = batch_y.reshape(B*T, )

                loss = self.loss_function.calculate(output, y_reshape)
                self.loss_vals.append(loss)

                if print_every is not None and not i % print_every:
                    print(f'===== EPOCH : {i} ===== LOSS : {loss} =====')
                
            else:
                self.forward(X, training=True)
                self.backward(self.output, y)

                self.optimizer.pre_update_parameters()
                self.optimizer.update_parameters(self)
                self.optimizer.post_update_parameters()

                B, T, C = self.output.shape
                output = self.output.reshape(B*T, C)
                y_reshape = y.reshape(B*T, )

                loss = self.loss_function.calculate(output, y_reshape)
                self.loss_vals.append(loss)

                if print_every is not None and not i % print_every:
                    print(f'===== EPOCH : {i} ===== LOSS : {loss} =====')
                
def generate(model, idx, context_window, max_new_tokens=100):
        
        for _ in range(max_new_tokens):

            current_context = idx[:, -context_window:]

            model.forward(current_context, training=False)
    
            logits = model.output

            probs = logits[:, -1, :].reshape(-1)

            #print(probs)

            idx_next = np.argmax(np.random.multinomial(n=1, pvals=probs, size=1), axis=1).reshape(1, 1)

            idx = np.concatenate((idx, idx_next), axis=1)

        return idx

# Shakespeare

In [4]:
with open('./data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [5]:
print(f'Characters: {len(text)}')

Characters: 1115394


In [6]:
print(f'{text[:100]}')

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [7]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f'All characters: {"".join(chars)}')
print(f'Vocab size: {vocab_size}')

All characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size: 65


In [8]:
encoded_dict = {chars[i]: i for i in range(vocab_size)}
decoded_dict = {i: chars[i] for i in range(vocab_size)}

def encode(s: str) -> list[int]:
    return [encoded_dict[c] for c in s]

def decode(s: list[int]) -> str:
    return "".join([decoded_dict[c] for c in s])

print(encode("test string"))
print(decode(encode("test string")))

[58, 43, 57, 58, 1, 57, 58, 56, 47, 52, 45]
test string


In [9]:
data = encode(text)
data = np.array(data, dtype=np.int32)
print(data.shape)

(1115394,)


# Train test split

In [10]:
n = int(0.9*len(data))
X_train, X_test = data[:n], data[n:]
print(f'n: {n}\nX_train: {X_train.shape}\nX_test: {X_test.shape}')

n: 1003854
X_train: (1003854,)
X_test: (111540,)


In [11]:
def create_sequences(data, seq_len = 8):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+1:i+seq_len+1])
    return np.array(X), np.array(y)

# Creating sequences

In [12]:
seq_len = 256

X_train, y_train = create_sequences(X_train, seq_len)
print(f'{X_train.shape}, {y_train.shape}')

(1003598, 256), (1003598, 256)


In [13]:
print(X_train[0, :5])
print(y_train[0, :5])

[18 47 56 57 58]
[47 56 57 58  1]


In [14]:
def get_random_batch(X, y, batch_size):
    idx = np.random.randint(0, len(X), size=(batch_size, ))
    return X[idx], y[idx]

In [15]:
np.random.seed(1337)

vocab_size = len(chars)
batch_size = 64

print(f'Vocab size: {vocab_size}, block size: {seq_len}')

n_embed = 192
n_head = 6
n_dec_layers = 3
dim_ff = 768
dropout = 0.2
eps = 1e-5

epochs = 5000
lr = 3e-4

loss = CCE_Loss(from_logits=False)
optimizer = Optimizer_Adam(learning_rate=lr, decay=0., clip_grad=False)

model = TransformerDecoderModel(vocab_size=vocab_size, 
                                block_size=seq_len, 
                                n_embed=n_embed, 
                                n_head=n_head,
                                n_dec_layers=n_dec_layers,
                                dim_ff=dim_ff, 
                                dropout=dropout,
                                layer_norm_eps=eps,
                                loss_function=loss, 
                                optimizer=optimizer)

model.train(X_train, y_train, print_every=1, epochs=epochs, batch_size=batch_size)

Vocab size: 65, block size: 256
===== EPOCH : 0 ===== LOSS : 5.275291045585364 =====
===== EPOCH : 1 ===== LOSS : 4.79719466060047 =====
===== EPOCH : 2 ===== LOSS : 4.421934033102364 =====
===== EPOCH : 3 ===== LOSS : 4.158165805771181 =====
===== EPOCH : 4 ===== LOSS : 4.007600603332582 =====
===== EPOCH : 5 ===== LOSS : 3.9002674431126203 =====
===== EPOCH : 6 ===== LOSS : 3.8369364634956415 =====
===== EPOCH : 7 ===== LOSS : 3.796146994357067 =====
===== EPOCH : 8 ===== LOSS : 3.7530794146518156 =====
===== EPOCH : 9 ===== LOSS : 3.714150947291322 =====
===== EPOCH : 10 ===== LOSS : 3.6583125241646313 =====
===== EPOCH : 11 ===== LOSS : 3.6807107870122557 =====
===== EPOCH : 12 ===== LOSS : 3.67460123170824 =====
===== EPOCH : 13 ===== LOSS : 3.6067373630455353 =====
===== EPOCH : 14 ===== LOSS : 3.622254429045319 =====
===== EPOCH : 15 ===== LOSS : 3.612972563216527 =====
===== EPOCH : 16 ===== LOSS : 3.6497945114845405 =====
===== EPOCH : 17 ===== LOSS : 3.5703477137777733 =====


KeyboardInterrupt: 

In [18]:
print(decode(generate(model, np.zeros((1, 1), dtype=np.int32), seq_len, max_new_tokens=500)[0].tolist()))


lheatnhHIm
tf
yJ,f, ooo
rhw ao  CrwiihleeeIrdt  oUs CfaraoAweeyeG eyssdrv'go aai sodfptIoeuiwsilar:sces
 Oa ty,egaIesroloEd
yehod HoGco;eea yeh?Knnke ;r n yi ;ioh :drr,o: dw r
,  Std unnGro, e . ps
enhp
uavt  t uaudo
sIvtoo ttF .E Tesr:udlfo  r,uierpc
y;u ioeyinmeeiwyhumeb
bf o ,u gtte to  ohhe  st orsBooereeesee hes gst pmY l

 i  FPawma  os.  yasuAtmhi    .sYS n
echiheedeln
pen.ouarhtho nuroutItiti?dhee,b
uoeN
cfinapkBleanpmseih 
 An:oe' hsniinnee vtirei.mwoe  mI,nk r!nae  n io gritc lsilne ek
